# Descritive

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

import sys, os, time, getpass

sys.path.append("/home/tatiane/lib/")
import pessoal
from pessoal import *

spark = SparkSession.builder \
    .appName("IBGE Processing") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [4]:
PATH = '/home/tatiane/repositorios/brazil-health-data-project/data_pipeline'
pnad_populacao_path = f"{PATH}/data/processed/ibge_populacao/ibge_populacao2.parquet"
ibge_populacao = spark.read.parquet(pnad_populacao_path)

In [5]:
pessoal.completudeSchema(ibge_populacao)

[Stage 2:>                                                          (0 + 1) / 1]

Qtd. registros: 22396 | Quantidade de colunas:  6
root
 |-- municipio: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- cod_uf: long (nullable = true)
 |-- ano: long (nullable = true)
 |-- populacao: long (nullable = true)
 |-- cod_municipio: long (nullable = true)



In [6]:
pessoal.completudeVar(ibge_populacao)

Resumo do DataFrame:


,coluna,total,duplicados,nulos,valores_unicos
0,ano,22396,22392,0,4
1,uf,22396,22368,112,28
2,cod_uf,22396,22368,4,28
3,municipio,22396,17078,0,5318
4,cod_municipio,22396,16824,112,5572
5,populacao,22396,5762,8,16634


In [7]:
for col in ibge_populacao.columns:
    ibge_populacao.groupBy(col).count().show()

+--------------------+-----+
|           municipio|count|
+--------------------+-----+
|        Assis Brasil|    4|
|    Senador Guiomard|    4|
|          Itaguatins|    4|
|      Benedito Leite|    4|
|Santo Antônio dos...|    4|
|  São João dos Patos|    4|
|               Apodi|    4|
|          Livramento|    4|
|              Maruim|    4|
|              Itatim|    4|
|              Utinga|    4|
|           Carrancas|    4|
|           Fronteira|    4|
|            Guidoval|    4|
|         Piranguinho|    4|
| Presidente Olegário|    4|
|            Rio Novo|    4|
|            Araruama|    4|
|Boa Esperança do Sul|    4|
|           Buritizal|    4|
+--------------------+-----+
only showing top 20 rows
+----+-----+
|  uf|count|
+----+-----+
|  SC| 1180|
|  RO|  208|
|  PI|  896|
|  AM|  248|
|  RR|   60|
|  GO|  984|
|NULL|  112|
|  TO|  556|
|  MT|  568|
|  SP| 2580|
|  PB|  892|
|  ES|  312|
|  RS| 1988|
|  MS|  316|
|  AL|  408|
|  MG| 3412|
|  PA|  576|
|  BA| 1668|
|  SE|

In [8]:
numeric_cols = [
    c for c,t in ibge_populacao.dtypes
    if t in ["int","bigint","double","float"]
]

In [9]:
ibge_populacao.select(numeric_cols).summary().show()

[Stage 131:>                                                        (0 + 1) / 1]

+-------+-----------------+-----------------+-----------------+-----------------+
|summary|           cod_uf|              ano|        populacao|    cod_municipio|
+-------+-----------------+-----------------+-----------------+-----------------+
|  count|            22392|            22396|            22388|            22284|
|   mean|32.36530903894248|           2019.5|  75372.106128283| 3253922.53275893|
| stddev|9.854408359828447|1.118058950158031|860040.3526229631|985066.8709661327|
|    min|               11|             2018|              771|          1100015|
|    25%|               25|             2018|             5464|          2512101|
|    50%|               31|             2019|            11708|          3146305|
|    75%|               41|             2020|            25868|          4119202|
|    max|               53|             2021|         46649132|          5300108|
+-------+-----------------+-----------------+-----------------+-----------------+

